# PyADM1ODE Calibration Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/PyADM1ODE_calibration/blob/main/notebooks/calibration_tutorial.ipynb)

This notebook demonstrates the complete calibration workflow for a biogas plant model using **PyADM1ODE_calibration**.

## 1. Setup Environment

First, we need to install the package and its dependencies.

In [ ]:
!sudo apt-get update
!pip install git+https://github.com/dgaida/PyADM1ODE_calibration.git

## 2. Load and Prepare Data

We use the built-in `MeasurementData` loader to handle CSV files.

In [ ]:
import pandas as pd
import numpy as np
from pyadm1ode_calibration.io.loaders import MeasurementData

# Create dummy data for demonstration
dates = pd.date_range("2024-01-01", periods=24 * 7, freq="h")
data = pd.DataFrame(
    {
        "Q_ch4": 500 + 50 * np.sin(np.arange(len(dates)) / 10) + np.random.normal(0, 10, len(dates)),
        "pH": 7.0 + 0.1 * np.cos(np.arange(len(dates)) / 15) + np.random.normal(0, 0.02, len(dates)),
        "Q_sub_maize": 11.4,
        "Q_sub_manure": 6.1,
    },
    index=dates,
)
data.index.name = "timestamp"
data.to_csv("tutorial_measurements.csv")

measurements = MeasurementData.from_csv("tutorial_measurements.csv")
print(f"Loaded {len(measurements)} hours of data.")

## 3. Define Plant Model

For this tutorial, we assume a plant model is already available. In a real scenario, you would load your `pyadm1ode.Plant` instance.

In [ ]:
from pyadm1 import BiogasPlant, Feedstock
from pyadm1.configurator.plant_configurator import PlantConfigurator

# 1. Create feedstock from substrate names
feedstock = Feedstock(
    substrates=["maize_silage_milk_ripeness", "swine_manure"],
    feeding_freq=24,
    total_simtime=10,
)

# 2. Build the plant
plant = BiogasPlant("Calibration Tutorial Plant")
configurator = PlantConfigurator(plant, feedstock)

# 3. Define substrate feed [m^3/d]
Q_substrates = [11.4, 6.1]

# 4. Add a digester
configurator.add_digester(
    digester_id="main_digester",
    V_liq=1200.0,
    V_gas=216.0,
    T_ad=315.15,
    name="Main Digester",
    Q_substrates=Q_substrates,
)

# 5. Initialize
plant.initialize()

## 4. Run Initial Calibration

We use the `InitialCalibrator` with Differential Evolution.

In [ ]:
from pyadm1ode_calibration.calibration import InitialCalibrator

calibrator = InitialCalibrator(plant)

# Run calibration for 2 parameters
# Note: In a real simulation, this might take several minutes.
result = calibrator.calibrate(
    measurements=measurements,
    parameters=["k_dis", "k_hyd_ch"],
    objectives=["Q_ch4", "pH"],
    max_iterations=5,  # Low for demonstration
)

print("Success:", result.success)
print("Optimized Parameters:", result.parameters)
print("RMSE:", result.objective_value)

## 5. Visualize Results

You can now compare the simulation with the measurements.

In [ ]:
import matplotlib.pyplot as plt

print(result)
print(result.history)

# Plotting the objective value history
plt.figure(figsize=(10, 5))
plt.plot([item['objective'] for item in result.history])
plt.title("Optimization History")
plt.xlabel("Iteration")
plt.ylabel("Cost Function Value")
plt.show()